# 08_fusion_dev — Lightsheet Fusion Parameter Sweep

**Kernel:** Python (spatialdata_2025)
**GPU execution:** Cell 2 runs 07_direct_fuse.py in lightsheet_env via subprocess (CUDA loaded inside subprocess).

## Workflow
1. Edit `PARAMS` in Cell 1 to set substack range and fusion parameters.
2. Run Cell 2 to execute fusion (≈16 min on A100 GPU node).
3. Run Cells 3–5 for diagnostics.
4. Run Cell 6 to print `best_params` for copy-paste into `07_direct_fuse.sh`.

**Prerequisites:** Run this notebook on an interactive GPU node (`srun --gres=gpu:A100:1 --pty bash` then launch JupyterLab).
If running from a login node, Cell 2 will fall back to CPU (NumPy) and take ~3–4 hours — not recommended.

In [ ]:
# ── PARAMETERS ─────────────────────────────────────────────────────────────
# Edit values here, then re-run Cell 2 to fuse with updated parameters.
# Parameters to sweep for seam/artifact reduction:
#   taper_px:      64 (current, visible seams) → try 96, 128, 144
#   sigma_frac:    0.3 (current) → try 0.2, 0.4, 0.5
#   ncc_threshold: 0.05 (current) → try 0.01, 0.1, 0.2
#   fusion_axis:   2=X (correct for KL018 Z.1 dual-side illumination)

PARAMS = {
    "z_start":       700,    # inclusive, first Z of substack
    "z_end":         728,    # exclusive, last Z of substack (100 planes)
    "sigma_frac":    0.8,    # Gaussian blend sigma fraction
    "taper_px":      128,    # cosine taper width in pixels (288 px overlap → 128 = 44%)
    "ncc_threshold": 0.5,   # NCC quality threshold for phase-correlation
    "fusion_axis":   2,      # 2=X axis (dual-side illumination along X for KL018)
    "skip_refine":   False,  # set True to skip NCC refinement (use raw stage positions)
    "z_chunk":       64,     # Z planes per processing slab
    "workers":       8,      # parallel CZI read threads
}

# ── PATHS ───────────────────────────────────────────────────────────────────
LIGHTSHEET_ENV = "/vast/scratch/users/kriel.j/lightsheet_env"
SCRIPT         = "/vast/projects/BCRL_Multi_Omics/scripts/lightsheet_pipeline/07_direct_fuse.py"
CZI_PATH       = "/vast/scratch/users/kriel.j/KL018_lightsheet/KL018_85_D7_CT2AvIII_Overview.czi"
OUT_DIR        = "/vast/scratch/users/kriel.j/KL018_lightsheet"
OUT_ZARR       = f"{OUT_DIR}/substack_z{PARAMS['z_start']}_{PARAMS['z_end']}.zarr"

print("PARAMS:")
for k, v in PARAMS.items():
    print(f"  {k:20s} = {v}")
print(f"\nOutput zarr: {OUT_ZARR}")

In [ ]:
import subprocess, sys

# Build CLI argument list from PARAMS
cli_args = [
    "--czi",            CZI_PATH,
    "--out",            OUT_ZARR,
    "--z-start",        str(PARAMS["z_start"]),
    "--z-end",          str(PARAMS["z_end"]),
    "--sigma-frac",     str(PARAMS["sigma_frac"]),
    "--taper-px",       str(PARAMS["taper_px"]),
    "--ncc-threshold",  str(PARAMS["ncc_threshold"]),
    "--fusion-axis",    str(PARAMS["fusion_axis"]),
    "--z-chunk",        str(PARAMS["z_chunk"]),
    "--workers",        str(PARAMS["workers"]),
]
if PARAMS["skip_refine"]:
    cli_args.append("--skip-refine")

# Use bash -c with module load CUDA/12.1 to ensure CuPy finds the correct CUDA toolkit.
# Mirrors the pattern in 07_direct_fuse.sh lines 41-44.
python_in_env = f"{LIGHTSHEET_ENV}/bin/python"
args_str = " ".join(cli_args)
bash_cmd = f"module load CUDA/12.1 2>/dev/null || true; {python_in_env} {SCRIPT} {args_str}"

print("Running fusion...")
print(f"  cmd: bash -c '{bash_cmd[:120]}...'")

result = subprocess.run(
    ["bash", "-c", bash_cmd],
    capture_output=True,
    text=True,
)

# Print last 4000 chars of stdout (progress bars, chunk completions)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode != 0:
    print("\n── STDERR ──────────────────────────────────")
    print(result.stderr[-2000:])
    raise RuntimeError(f"07_direct_fuse.py exited with code {result.returncode}")
else:
    print(f"\nDone. Zarr written to: {OUT_ZARR}")

In [ ]:
import zarr
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

print(f"Opening {OUT_ZARR} ...")
z = zarr.open(OUT_ZARR, "r")
print(f"  Shape (TCZYX): {z.shape}   dtype: {z.dtype}   chunks: {z.chunks}")

# Mid-local-Z index (50 for 100-slice substack)
mid_loc = z.shape[2] // 2
mid_abs = PARAMS["z_start"] + mid_loc
print(f"  Displaying Z={mid_abs} (local index {mid_loc}), channel 0, 1/8 downsample")

# Load at 1/8 downsample to keep array ~3 MB
plane = z[0, 0, mid_loc, ::8, ::8]

p1  = np.percentile(plane, 1)
p99 = np.percentile(plane, 99)

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(plane, cmap="gray", vmin=p1, vmax=p99, interpolation="nearest")
ax.set_title(
    f"Mid-Z MIP mosaic  —  Z={mid_abs} (local {mid_loc}), CH0, 1/8 downsample\n"
    f"taper_px={PARAMS['taper_px']}  sigma_frac={PARAMS['sigma_frac']}  "
    f"ncc_threshold={PARAMS['ncc_threshold']}"
)
ax.set_xlabel("X (pixels ÷ 8)")
ax.set_ylabel("Y (pixels ÷ 8)")
plt.colorbar(im, ax=ax, label="intensity (uint16)", fraction=0.025)
plt.tight_layout()
plt.show()
print(f"  pixel range [p1, p99]: [{p1:.0f}, {p99:.0f}]  array shape: {plane.shape}")

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

ncc_path = f"{OUT_DIR}/ncc_scores.json"
print(f"Loading NCC scores from: {ncc_path}")
with open(ncc_path) as f:
    ncc_data = json.load(f)

quality   = np.array(ncc_data["quality_matrix"])   # (n_tiles, n_tiles)
tile_pos  = ncc_data["tile_positions_refined"]      # list of {M, x, y, w, h} absolute coords
mid_z_ncc = ncc_data["mid_z"]
ncc_thresh = ncc_data["ncc_threshold"]

print(f"  {len(tile_pos)} tiles  |  mid_z used for NCC: {mid_z_ncc}  |  threshold: {ncc_thresh}")
print(f"  quality matrix shape: {quality.shape}  "
      f"range: [{quality.min():.3f}, {quality.max():.3f}]")

# Canvas origin — compute first so centers are in canvas-relative space
canvas_ox = min(p["x"] for p in tile_pos)
canvas_oy = min(p["y"] for p in tile_pos)

# Build tile-center lookup in canvas-relative coordinates
centers = {p["M"]: (p["x"] - canvas_ox + p["w"] // 2, p["y"] - canvas_oy + p["h"] // 2) for p in tile_pos}

fig, ax = plt.subplots(figsize=(12, 10))
norm = Normalize(vmin=0, vmax=max(quality.max(), 1e-6))
cmap = plt.cm.RdYlGn

# Plot a colored square at each seam midpoint between adjacent tile pairs
n = len(tile_pos)
for i in range(n):
    for j in range(i + 1, n):
        q = quality[i][j]
        if q > 0:
            cx = (centers[i][0] + centers[j][0]) / 2
            cy = (centers[i][1] + centers[j][1]) / 2
            ax.plot(cx, cy, "s", color=cmap(norm(q)), markersize=12, alpha=0.85)

# Draw tile outlines in canvas-relative coordinates
for p in tile_pos:
    rect = plt.Rectangle(
        (p["x"] - canvas_ox, p["y"] - canvas_oy), p["w"], p["h"],
        fill=False, edgecolor="gray", linewidth=0.7, linestyle="--"
    )
    ax.add_patch(rect)
    ax.text(
        p["x"] - canvas_ox + p["w"] // 2,
        p["y"] - canvas_oy + p["h"] // 2,
        str(p["M"]), ha="center", va="center", fontsize=7, color="white", alpha=0.6
    )

ax.set_xlim(0, max(p["x"] - canvas_ox + p["w"] for p in tile_pos))
ax.set_ylim(max(p["y"] - canvas_oy + p["h"] for p in tile_pos), 0)
ax.set_aspect("equal")
fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, label="NCC quality score")
ax.set_title(
    f"Seam quality heatmap  —  NCC threshold={ncc_thresh}\n"
    f"Green=strong overlap, Red=weak/no overlap (squares at seam midpoints)"
)
ax.set_xlabel("Canvas X (px)")
ax.set_ylabel("Canvas Y (px)")
plt.tight_layout()
plt.show()

In [ ]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
import json

# Reload zarr handle if kernel was restarted
z_arr = zarr.open(OUT_ZARR, "r")
mid_loc = z_arr.shape[2] // 2

# Load full mid-Z plane for tile crops (188 MB, uint16 — OK for spatialdata_env_2)
print(f"Loading mid-Z plane (Z local={mid_loc}, shape {z_arr.shape[3]}×{z_arr.shape[4]}) ...")
plane_full = z_arr[0, 0, mid_loc, :, :]   # (canvas_H, canvas_W) uint16

# Use tile positions from ncc_scores.json (absolute canvas coords)
# Offset to canvas-relative coords using min(x) and min(y) as origin
canvas_ox_abs = min(p["x"] for p in tile_pos)
canvas_oy_abs = min(p["y"] for p in tile_pos)

fig, ax = plt.subplots(figsize=(14, 5))
n_plotted = 0
for p in tile_pos:
    x0 = p["x"] - canvas_ox_abs
    y0 = p["y"] - canvas_oy_abs
    x1 = x0 + p["w"]
    y1 = y0 + p["h"]

    # Clip to canvas bounds (refined canvas may be slightly smaller than max extent)
    x0c = max(x0, 0)
    y0c = max(y0, 0)
    x1c = min(x1, plane_full.shape[1])
    y1c = min(y1, plane_full.shape[0])
    if x1c <= x0c or y1c <= y0c:
        continue

    tile_crop = plane_full[y0c:y1c, x0c:x1c]
    # Mean along Y axis → 1D profile across X showing illumination gradient
    profile = tile_crop.mean(axis=0)
    ax.plot(profile, alpha=0.45, linewidth=0.9, label=f"M{p['M']}")
    n_plotted += 1

ax.set_xlabel("X pixel within tile (0 = left edge)")
ax.set_ylabel("Mean intensity (counts)")
ax.set_title(
    f"Per-tile illumination X-profile  —  {n_plotted} tiles plotted\n"
    f"Flat line = uniform illumination. Gradient = dual-side fusion issue (adjust sigma_frac or fusion_axis)."
)
# Only show legend if ≤10 tiles (30 tiles makes legend unreadable)
if n_plotted <= 10:
    ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()
print(f"  Plotted {n_plotted}/{len(tile_pos)} tiles")
del plane_full  # free ~188 MB

In [ ]:
# ── COPY-PASTE OUTPUT FOR 07_direct_fuse.sh ─────────────────────────────────
# After visual inspection of Cells 3–5, update PARAMS in Cell 1 to the
# values that produced the best fusion quality, then run this cell.
# Copy the printed lines into 07_direct_fuse.sh to replace the current flags.

best_params = {
    "sigma_frac":     PARAMS["sigma_frac"],
    "taper_px":       PARAMS["taper_px"],
    "ncc_threshold":  PARAMS["ncc_threshold"],
    "fusion_axis":    PARAMS["fusion_axis"],
    "z_chunk":        PARAMS["z_chunk"],
    "workers":        PARAMS["workers"],
    # skip_refine intentionally excluded — use only if NCC is failing
}

print("# ── Paste into 07_direct_fuse.sh (replace current python call flags) ──────")
print(f'python "${{SCRIPT_DIR}}/07_direct_fuse.py" \\')
print(f'    --czi      "$CZI_ARG" \\')
for k, v in best_params.items():
    flag = k.replace("_", "-")
    print(f"    --{flag:<18} {v} \\")
if PARAMS["skip_refine"]:
    print("    --skip-refine \\")
print("    ;")

print("\n# ── Full best_params dict (for records) ────────────────────────────────────")
print("best_params =", best_params)